Tokenization stats:  
Max: 15732  
Average: 1483  
trainDatasets/summeries_cross.csv

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

model_name = "imvladikon/het5_small_summarization"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load your dataset
data = load_dataset("csv", data_files="trainDatasets/summeries_cross.csv")  # or use 'json'

# Tokenize
def preprocess(example):
    inputs = tokenizer(
        example["conversation"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    targets = tokenizer(
        example["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_data = data.map(preprocess, batched=True)

Map: 100%|██████████| 21298/21298 [00:41<00:00, 512.34 examples/s]


In [2]:
#'input_ids', 'attention_mask', 'labels'
tokenized_data["train"]["conversation"][0]

'פונה: אל תשטירו אותי לבד בבקשה אני מתחננת[SEP]פונה: אני סובלת לבד ואין אף אחד שיהיה איתי עכשיו  ואני מתחרפנת[SEP]נציג: שלום, אנחנו כאן עד <TIME>.. שומעת שאת נסערת הלילה.. תוכלי לבחור שנדבר עד אז או לעלות במוצש ולממש שיחה מלאה.. [SEP]נציג: תרצי שנשוחח מעט אולי לפרוק קצת את הסערה שמשתוללת..?[SEP]פונה: לא לא עד מוצש אני ימות אני חייבת לדבר עם מישו[SEP]פונה: התכוונתי שעד מוצש זה הרבה זמן[SEP]פונה: אני מרגישה רע כל כך רע[SEP]נציג: נוכל להיות יחד ולדבר בזמן שנותר לנו.. תרצי לשתף אותי קצת כדי שאבין יותר?[SEP]פונה: אני בדיכאון חזק א[SEP]פונה: בשבוע האחרון היו לי מחשבות אובדניות זה פשוט לא יוצא לי מהראש[SEP]פונה: נראה לי שהדיכאון החמיר לי ממש[SEP]פונה: זה פשוט מחשבה כפייתית כזאת לא יודעת[SEP]נציג: שומעת את הכאב בין המילים, מעין תחושה של חוסר שליטה על המחשבות שלך.. ואילו את רק מבקשת שקט[SEP]פונה: אוקיי בסדר[SEP]פונה: אז לילה טוב אני מניחה?[SEP]נציג: יקרה, עלינו לסיים את השיחה.. עם זאת נשמע שאת עדיין נסערת מעט ואני מציעה לך אולי לפנות למוקד של <שם_> שעדיין פעיל. כמו כן, אני מאירופה לך כאן לינק ל

In [3]:
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Optional: Add sentence splitting for ROUGE consistency
    decoded_preds = ["\n".join(pred.strip().split(".")) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split(".")) for label in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # Only keep F1 scores for clarity
    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }

In [4]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

training_args = Seq2SeqTrainingArguments(
    output_dir="./finetuned_het5",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    learning_rate=5e-5,
    predict_with_generate=True,
    logging_dir="./logs",
    fp16=True,  # if using GPU with mixed precision
    save_total_limit=2,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["train"],  # use a separate eval split ideally
    tokenizer=tokenizer,
    data_collator=data_collator
)

/tmp/ipykernel_1816715/1262026576.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [5]:
trainer.train()

ValueError: You have to specify either decoder_input_ids or decoder_inputs_embeds